## Parte 2 · Arquitecto Cloud: benchmark, entorno y conectividad AWS/GCP

**Metodología.** Se procesan los mismos CSV con las mismas reglas de limpieza y agregación:
- **Pandas secuencial:** lee cada archivo en bloques de `CHUNKSIZE` filas, agrega cada bloque y combina los parciales.
- **Dask paralelo:** `dd.read_csv` con particiones de `BLOCKSIZE`, ejecutado con el scheduler `SCHEDULER` y `NUM_WORKERS` workers.
- Cada método se mide `REPEATS` veces, alternando Pandas y Dask. Se reporta la mediana y todos los tiempos. La primera pasada puede verse afectada por la caché del disco.
- Antes de aceptar el resultado se verifica la equivalencia: mismos grupos, mismos conteos y unidades, y montos iguales dentro de la tolerancia de punto flotante.

**Alcance de las conclusiones.** Las conclusiones sobre velocidad y sobre RAM se basan solo en lo medido, en el equipo donde se ejecutó. La comparación con la RAM usa el tamaño del CSV en disco frente a la RAM total y no mide el pico de memoria de cada método.

**A completar por el Arquitecto Cloud:**
- Equipo donde se midió y condiciones (aplicaciones abiertas, modo de energía): [completar]
- Interpretación del resultado: [completar]

**Conectividad y entorno cloud (AWS/GCP):**
- Plataforma y recurso desplegado (tipo, región), o indicar que no se desplegó: [completar]
- Cómo se conecta el recurso a MongoDB Atlas (IP autorizada en Network Access): [completar]
- Cómo se ejecuta el cuaderno en ese recurso: [completar]
- Capturas del recurso en ejecución (se adjuntan al informe): [completar]

In [1]:
def pandas_sequential(paths, chunksize=CHUNKSIZE):
    files = sorted(glob.glob(paths))
    if not files:
        raise FileNotFoundError(f'No hay CSV para {paths}')
    partials, rows_read = [], 0
    for f in files:
        for chunk in pd.read_csv(f, chunksize=chunksize, usecols=USECOLS, dtype=DTYPES):
            rows_read += len(chunk)
            partials.append(clean_frame(chunk).groupby(GROUPS)[['amount', 'n', 'units']].sum())
    agg = pd.concat(partials).groupby(level=GROUPS).sum()
    result = finalize(agg.reset_index())
    return result, {'rows_read': int(rows_read), 'rows_kept': int(result['event_count'].sum())}

def check_equivalence(p, d, p_info, d_info, manifest):
    checks = {'filas leídas iguales': p_info['rows_read'] == d_info['rows_read'] == manifest['records'],
              'filas válidas iguales': p_info['rows_kept'] == d_info['rows_kept'] == manifest['expected_valid_rows'],
              'mismo número de grupos': len(p) == len(d)}
    max_rel = None
    if checks['mismo número de grupos']:
        checks['mismas llaves (región, categoría, mes)'] = bool(p[GROUPS].equals(d[GROUPS]))
        if checks['mismas llaves (región, categoría, mes)']:
            checks['event_count idéntico'] = bool(np.array_equal(p['event_count'], d['event_count']))
            checks['total_units idéntico'] = bool(np.array_equal(p['total_units'], d['total_units']))
            checks['total_amount igual (rtol 1e-9)'] = bool(np.allclose(p['total_amount'], d['total_amount'], rtol=1e-9, atol=1e-6))
            checks['avg_amount igual (rtol 1e-9)'] = bool(np.allclose(p['avg_amount'], d['avg_amount'], rtol=1e-9, atol=1e-9))
            max_rel = float((np.abs(p['total_amount'] - d['total_amount']) / d['total_amount']).max())
    return checks, max_rel

# --- Medición: se alternan Pandas y Dask ---
pandas_times, dask_times = [], []
for r in range(REPEATS):
    t0 = time.perf_counter(); pandas_result, pandas_info = pandas_sequential(paths); pandas_times.append(time.perf_counter() - t0)
    t0 = time.perf_counter(); dask_result, dask_info = dask_parallel(paths);          dask_times.append(time.perf_counter() - t0)
    print(f'Repetición {r+1}/{REPEATS}: pandas={pandas_times[-1]:.2f}s  dask={dask_times[-1]:.2f}s')

checks, max_rel = check_equivalence(pandas_result, dask_result, pandas_info, dask_info, manifest)
for k, v in checks.items():
    print('OK ' if v else 'FALLO', k)
print('Máxima diferencia relativa en total_amount:', max_rel)
assert all(checks.values()), 'Pandas y Dask NO son equivalentes; no se guarda el benchmark.'


NameError: name 'CHUNKSIZE' is not defined

In [ ]:
pandas_med, dask_med = statistics.median(pandas_times), statistics.median(dask_times)
speedup = pandas_med / dask_med if dask_med else None
hw = hardware_info()
input_gb = manifest['input_bytes'] / 1024**3

if speedup is None:
    conclusion = 'No se pudo calcular el speedup.'
elif speedup > 1.10:
    conclusion = f'En esta medición Dask fue más rápido que Pandas (speedup {speedup:.2f}x).'
elif speedup < 0.90:
    conclusion = f'En esta medición Dask fue más lento que Pandas (speedup {speedup:.2f}x).'
else:
    conclusion = f'En esta medición no hubo diferencia relevante entre Pandas y Dask (speedup {speedup:.2f}x).'

benchmark = {
    'fecha_ejecucion_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'versiones': {'python': sys.version.split()[0], 'pandas': pd.__version__, 'dask': dask.__version__, 'numpy': np.__version__},
    'hardware': hw,
    'datos': {'filas_generadas': manifest['records'], 'filas_validas': dask_info['rows_kept'],
              'archivos_csv': manifest['files'], 'bytes_entrada': manifest['input_bytes'],
              'entrada_gb': round(input_gb, 3), 'grupos_agregados': len(dask_result),
              'entrada_supera_ram_total': bool(input_gb > hw['ram_total_gb']),
              'nota_ram': 'Compara tamaño del CSV en disco con la RAM total; no mide el pico de memoria.'},
    'configuracion': {'chunksize_pandas': CHUNKSIZE, 'blocksize_dask': BLOCKSIZE, 'particiones_dask': dask_info['npartitions'],
                      'scheduler': SCHEDULER, 'num_workers': NUM_WORKERS, 'repeticiones': REPEATS},
    'tiempos': {'pandas_segundos': [round(t, 3) for t in pandas_times], 'dask_segundos': [round(t, 3) for t in dask_times],
                'pandas_mediana': round(pandas_med, 3), 'dask_mediana': round(dask_med, 3)},
    'speedup_mediana': round(speedup, 3) if speedup else None,
    'conclusion_automatica': conclusion,
    'equivalencia': {**checks, 'max_diferencia_relativa_total_amount': max_rel},
}
(RESULTS_DIR / 'benchmark.json').write_text(json.dumps(benchmark, indent=2, ensure_ascii=False), encoding='utf-8')
dask_result.to_csv(RESULTS_DIR / 'agregados_dask.csv', index=False)   # mismo contenido que la Parte 1

print(json.dumps(benchmark, indent=2, ensure_ascii=False))
print('\nRAM vs entrada:', f"CSV {input_gb:.3f} GB vs RAM total {hw['ram_total_gb']} GB ->",
      'la entrada SUPERA la RAM total' if benchmark['datos']['entrada_supera_ram_total'] else 'la entrada NO supera la RAM total')

**Revisión cruzada de la Parte 2** · Revisor: Ingeniero de Datos ([completar nombre])
- [ ] Ejecutó el cuaderno completo en su máquina y comparó sus tiempos con los del Arquitecto Cloud (los tiempos cambian entre equipos).
- [ ] Comprobó que las reglas de limpieza son idénticas en ambos métodos.
- [ ] Revisó que las conclusiones del informe coinciden con `benchmark.json`.
- Observaciones: [completar]